# 선정 쪽수 전체 청크 → 적재 후보 패키지

전체 원문의 위치별 구조를 먼저 읽어 청크를 만든다. 기존 골든셋은 문서·쪽수·발췌로 재탐색하며 자동 변경하지 않는다. 본문 ID의 내용 변경과 행동 메타데이터를 검토하기 전에는 새 패키지를 공유 DB에 적재하지 않는다.


In [2]:
import json
import re
import sys
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from index.school.bundle import build_load_bundle
from index.school.dictionaries import load_dictionaries
from index.school.id_map import IdRegistry, document_node, load_manifest
from index.school.normalize import read_jsonl, write_jsonl
from preprocess.school.chunks import CHUNKER_VERSION, build_all_chunks
from preprocess.school.chunk_audit import citation_audit

NORMALIZED = ROOT / "data" / "interim" / "school" / "normalized_20261105T032224_612625Z"   # 이전 승인 행동. 새 경로에 따른 단계·조건 재검토 필요
books = load_dictionaries()
manifest = load_manifest(ROOT)
chunks, about = build_all_chunks(ROOT, manifest, books)
print("청크:", len(chunks), dict(Counter((c["document_id"], c["kind"]) for c in chunks)))
print("ABOUT:", len(about), dict(Counter(v[0] for v in about.values())))
lengths = [len(c["text"]) for c in chunks if c["kind"] == "paragraph"]
print("문단 길이 최소/중앙/최대:", min(lengths), sorted(lengths)[len(lengths) // 2], max(lengths))

청크: 1780 {('moe_school_manual_v3', 'table_row'): 170, ('moe_school_manual_v3', 'paragraph'): 434, ('gyeonggi_school_guide_2025', 'table_row'): 596, ('gyeonggi_school_guide_2025', 'paragraph'): 580}
ABOUT: 162 {'TUBERCULOSIS': 18, 'VARICELLA': 23, 'HFMD': 18, 'EKC': 12, 'MUMPS': 15, 'INFLUENZA': 16, 'PERTUSSIS': 15, 'SCARLET_FEVER': 17, 'MEASLES': 16, 'COVID19': 12}
문단 길이 최소/중앙/최대: 3 47 799


## 검사 1 — 골든 발췌 재탐색과 ID 이행 검토


In [4]:
golden = json.loads((ROOT / "data/specs/golden/school.json").read_text(encoding="utf-8"))
checks = citation_audit(golden["citations"], chunks)
display(pd.DataFrame(checks)[["chunk_id", "candidates", "old_id_valid"]])
print("발췌 재탐색:", sum(bool(c["candidates"]) for c in checks), "/", len(checks))
print("기존 ID의 발췌 유효:", sum(c["old_id_valid"] for c in checks), "/", len(checks))
assert checks and all(c["candidates"] for c in checks)
print("후보 매핑이며 골든셋·DB는 변경하지 않음. 기존 ID만으로 새 청크를 동일 자료로 판단하지 말 것.")


                                   chunk_id  ...  old_id_valid
0            moe_school_manual_v3:p0053:b01  ...         False
1            moe_school_manual_v3:p0054:b01  ...         False
2            moe_school_manual_v3:p0055:b01  ...          True
3   gyeonggi_school_guide_2025:p0166:t01r01  ...          True
4   gyeonggi_school_guide_2025:p0166:t01r01  ...          True
..                                      ...  ...           ...
68           moe_school_manual_v3:p0054:b01  ...         False
69           moe_school_manual_v3:p0053:b01  ...         False
70  gyeonggi_school_guide_2025:p0164:t01r09  ...          True
71        moe_school_manual_v3:p0053:t01r01  ...          True
72        moe_school_manual_v3:p0056:t01r02  ...          True

[73 rows x 3 columns]
발췌 재탐색: 73 / 73
기존 ID의 발췌 유효: 62 / 73
후보 매핑이며 골든셋·DB는 변경하지 않음. 기존 ID만으로 새 청크를 동일 자료로 판단하지 말 것.


## 검사 2 — 표 행 ID 장부 호환성(저장하지 않음)


In [6]:
registry = IdRegistry.load()
before = len(registry.data["chunks"])
for chunk in chunks:
    if chunk["kind"] == "table_row":
        registry.register_chunk(chunk)
print(f"표 행 장부 검증: {before} → {len(registry.data['chunks'])}. 파일 저장 없음.")
print("재분할된 본문은 migration.json의 내용 비교와 참조 이행 검토 필요")


표 행 장부 검증: 1139 → 1139. 파일 저장 없음.
재분할된 본문은 migration.json의 내용 비교와 참조 이행 검토 필요


## 적재 패키지

In [8]:
actions = read_jsonl(NORMALIZED / "normalized.jsonl")
documents = [document_node(d, manifest) for d in sorted({c["document_id"] for c in chunks})]
bundle = build_load_bundle(actions, chunks, documents, books, concept_policy="omit_link", chunk_diseases=about)
print("노드:", {k: len(v) for k, v in bundle["nodes"].items()})
print("관계:", dict(Counter(e["type"] for e in bundle["edges"])))
print("DB에 먼저 있어야 할 사전 노드:", len(bundle["dictionary_refs"]))
out_dir = ROOT / "data" / "interim" / "school" / datetime.now(UTC).strftime("full_%Y%m%dT%H%M%S_%fZ")
out_dir.mkdir(parents=True)
write_jsonl(out_dir / "official_chunks.jsonl", chunks)
(out_dir / "candidate_bundle.json").write_text(json.dumps(bundle, ensure_ascii=False, indent=1, allow_nan=False), encoding="utf-8")
(out_dir / "meta.json").write_text(json.dumps({"chunker_version": CHUNKER_VERSION, "status": "candidate_requires_review", "actions_from": NORMALIZED.name, "created_at": datetime.now(UTC).isoformat(),
    "chunks": len(chunks), "about": len(about)}, ensure_ascii=False, indent=1), encoding="utf-8")
print("저장:", out_dir.relative_to(ROOT))

노드: {'Document': 2, 'Chunk': 1780, 'SchoolAction': 438}
관계: {'HAS_CHUNK': 1780, 'ABOUT': 162, 'SUPPORTED_BY': 438, 'INSTANCE_OF': 269, 'DONE_BY': 306, 'FOR_DISEASE': 93}
DB에 먼저 있어야 할 사전 노드: 46
저장: data\interim\school\full_20261105T222142_685516Z
